<a href="https://colab.research.google.com/github/dr-ayesha-seecs/ir-course-exercises/blob/main/IR_Exercises_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Advanced Information Retrieval — Exercises 1–3 (Colab runner)

1. Run the setup cell to clone the repository.
2. Edit `ir-course-exercises/exerciseN/starter.py` in the Colab file browser (left sidebar) and complete the TODOs.
3. Run the cell for that exercise to see your output.

**Download your edited files before the session ends.** See `SETUP_FAQ.md`.

In [2]:
import os, sys
if not os.path.exists('ir-course-exercises') and not os.path.exists('documents'):
    !git clone -q https://github.com/dr-ayesha-seecs/ir-course-exercises.git
if os.path.exists('ir-course-exercises'):
    %cd ir-course-exercises
print(sorted(os.listdir('.')))

/content/ir-course-exercises
['.DS_Store', '.git', '.gitignore', 'IR_Exercises_Colab.ipynb', 'README.md', 'REPORT_TEMPLATE.md', 'SETUP_FAQ.md', 'documents', 'exercise1', 'exercise2', 'exercise3', 'requirements.txt', 'stopwords.txt']


## Exercise 1: Inverted index and Boolean retrieval

In [16]:
!{sys.executable} exercise1/starter.py

Loaded 20 documents

First 20 tokens of doc01: ['cat', 'sat', 'mat', 'cat', 'like', 'warm', 'mat', 'near', 'window', 'sleep', 'every', 'afternoon']

Posting lists:
  'cat': ['doc01', 'doc03', 'doc08', 'doc15', 'doc19']
  'dog': ['doc02', 'doc03', 'doc05', 'doc09', 'doc13', 'doc15', 'doc20']
  'sat': ['doc01', 'doc02', 'doc10', 'doc17']
  'mat': ['doc01', 'doc12', 'doc20']
  'log': ['doc02', 'doc10']

Boolean queries:
"cat" AND "sat"     : ['doc01']
"cat" OR "dog"      : ['doc01', 'doc02', 'doc03', 'doc05', 'doc08', 'doc09', 'doc13', 'doc15', 'doc19', 'doc20']
"sat" AND NOT "dog" : ['doc01', 'doc10', 'doc17']


In [15]:
import re
import os

# Writing the fully robust and corrected starter.py
updated_code = """
import os
import re

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DOCS_DIR = os.path.join(ROOT, "documents")
STOPWORDS_FILE = os.path.join(ROOT, "stopwords.txt")

def load_documents(docs_dir=DOCS_DIR):
    docs = {}
    for name in sorted(os.listdir(docs_dir)):
        if name.endswith(".txt"):
            with open(os.path.join(docs_dir, name), encoding="utf-8") as f:
                docs[name[:-4]] = f.read()
    return docs

def load_stopwords(path=STOPWORDS_FILE):
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            return {line.strip().lower() for line in f if line.strip()}
    return set()

STOPWORDS = load_stopwords()

def preprocess(text):
    # Lowercase and split into alphanumeric tokens
    tokens = re.findall(r"[a-z0-9]+", text.lower())
    # Remove stopwords
    tokens = [t for t in tokens if t not in STOPWORDS]

    # Apply simple stemming (remove trailing "s", "ing", "ed")
    stemmed_tokens = []
    for word in tokens:
        for suffix in ('ing', 'ed', 's'):
            if word.endswith(suffix) and len(word) > len(suffix) + 2:
                word = word[:-len(suffix)]
                break
        stemmed_tokens.append(word)
    return stemmed_tokens

def build_inverted_index(docs):
    index = {}
    for doc_id, text in sorted(docs.items()):
        terms = preprocess(text)
        for term_name in set(terms):
            if term_name not in index:
                index[term_name] = []
            index[term_name].append(doc_id)

    # Ensure postings lists are sorted
    for term_name in index:
        index[term_name] = sorted(list(set(index[term_name])))
    return index

def AND(index, t1, t2):
    # Support both list of doc IDs or term string
    p1 = t1 if isinstance(t1, list) else index.get(t1, [])
    p2 = t2 if isinstance(t2, list) else index.get(t2, [])

    # Linear-time intersection of sorted lists
    res = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            res.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            i += 1
        else:
            j += 1
    return res

def OR(index, t1, t2):
    # Support both list of doc IDs or term string
    p1 = t1 if isinstance(t1, list) else index.get(t1, [])
    p2 = t2 if isinstance(t2, list) else index.get(t2, [])

    # Linear-time union of sorted lists
    res = []
    i, j = 0, 0
    while i < len(p1) and j < len(p2):
        if p1[i] == p2[j]:
            res.append(p1[i])
            i += 1
            j += 1
        elif p1[i] < p2[j]:
            res.append(p1[i])
            i += 1
        else:
            res.append(p2[j])
            j += 1
    while i < len(p1):
        res.append(p1[i])
        i += 1
    while j < len(p2):
        res.append(p2[j])
        j += 1
    return res

def NOT(index, t, all_doc_ids):
    p = set(t if isinstance(t, list) else index.get(t, []))
    # Difference
    return sorted([d for d in all_doc_ids if d not in p])

def term(t):
    toks = preprocess(t)
    return toks[0] if toks else t.lower()

if __name__ == "__main__":
    docs = load_documents()
    print("Loaded", len(docs), "documents")

    # Part A
    print("\\nFirst 20 tokens of doc01:", preprocess(docs["doc01"])[:20])

    # Part B
    index = build_inverted_index(docs)
    print("\\nPosting lists:")
    for t in ['cat', 'dog', 'sat', 'mat', 'log']:
        print(f"  '{t}':", index.get(t, []))

    # Part C
    all_ids = sorted(docs.keys())
    t_cat, t_sat, t_dog = term('cat'), term('sat'), term('dog')
    print("\\nBoolean queries:")
    print(\'"cat" AND "sat"     :\', AND(index, t_cat, t_sat))
    print(\'"cat" OR "dog"      :\', OR(index, t_cat, t_dog))
    print(\'"sat" AND NOT "dog" :\', AND(index, t_sat, NOT(index, t_dog, all_ids)))
"""

with open('exercise1/starter.py', 'w', encoding='utf-8') as f:
    f.write(updated_code.strip())
print("Part C updated in starter.py successfully!")

Part C updated in starter.py successfully!


In [13]:
with open('exercise1/starter.py', 'r') as f:
    print(f.read())

"""
Exercise 1: Build Your First Search Engine
Inverted index + Boolean retrieval (AND, OR, NOT).

Run from the repository root:   python exercise1/starter.py
Fill in every block marked TODO. Do not change function names/signatures:
Exercises 2 and 3 reuse them.
"""
import os

ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
DOCS_DIR = os.path.join(ROOT, "documents")
STOPWORDS_FILE = os.path.join(ROOT, "stopwords.txt")


def load_documents(docs_dir=DOCS_DIR):
    """Return {doc_id: text}, e.g. {"doc01": "The cat sat ..."}. (Provided.)"""
    docs = {}
    for name in sorted(os.listdir(docs_dir)):
        if name.endswith(".txt"):
            with open(os.path.join(docs_dir, name), encoding="utf-8") as f:
                docs[name[:-4]] = f.read()
    return docs


def load_stopwords(path=STOPWORDS_FILE):
    """Return a set of stopwords. (Provided.)"""
    with open(path, encoding="utf-8") as f:
        return {line.strip().lower() for line in f if line.strip()}


STO

In [11]:
import string

# Let's read the existing content of starter.py first
with open('exercise1/starter.py', 'r') as f:
    content = f.read()

# We will replace the preprocess function in the file
preprocess_code = """def preprocess(text):
    \"\"\"Return a list of tokens.

    TODO:
      1. lowercase the text
      2. split into tokens (hint: drop punctuation first)
      3. remove stopwords (STOPWORDS)
      4. simple stemming: remove trailing \"s\", \"ing\", \"ed\"
    \"\"\"
    # 1. lowercase
    text = text.lower()

    # 2. drop punctuation and split into tokens
    translator = str.maketrans('', '', string.punctuation)
    text = text.translate(translator)
    raw_tokens = text.split()

    # 3. remove stopwords and 4. apply simple stemming
    tokens = []
    for token in raw_tokens:
        if token in STOPWORDS:
            continue

        # simple stemming
        if token.endswith('ing'):
            token = token[:-3]
        elif token.endswith('ed'):
            token = token[:-2]
        elif token.endswith('s'):
            token = token[:-1]

        if token and token not in STOPWORDS:
            tokens.append(token)

    return tokens"""

# We will replace the build_inverted_index function in the file
build_index_code = """def build_inverted_index(docs):
    \"\"\"Return {term: sorted list of doc_ids}.

    TODO: for each document, preprocess it and add its ID to each term's posting list.
    \"\"\"
    inverted_index = {}
    for doc_id, text in docs.items():
        tokens = preprocess(text)
        for token in set(tokens):
            if token not in inverted_index:
                inverted_index[token] = []
            inverted_index[token].append(doc_id)

    # Sort document IDs for each term's postings list
    for term_key in inverted_index:
        inverted_index[term_key] = sorted(list(set(inverted_index[term_key])))

    return inverted_index"""

# Replace preprocess implementation
start_idx = content.find("def preprocess(text):")
end_idx = content.find("# ---------------------------------------------------------------- Part B")
content = content[:start_idx] + preprocess_code + "\n\n" + content[end_idx:]

# Replace build_inverted_index implementation
start_idx = content.find("def build_inverted_index(docs):")
end_idx = content.find("# ---------------------------------------------------------------- Part C")
content = content[:start_idx] + build_index_code + "\n\n" + content[end_idx:]

# Save the file
with open('exercise1/starter.py', 'w') as f:
    f.write(content)

print("Functions successfully updated in exercise1/starter.py!")

Functions successfully updated in exercise1/starter.py!


## Exercise 2: TF-IDF and BM25

In [20]:
import math
import os

# Let's write the complete, robust implementation for exercise2/starter.py with escaped docstrings
exercise2_code = """
import os
import sys
import math

# Reuse preprocessing from Exercise 1
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from exercise1.starter import load_documents, preprocess, term

docs = load_documents()
all_doc_ids = sorted(docs.keys())
N = len(docs)

# ---------------------------------------------------------------- Part A
# Compute TF, IDF, and TF-IDF

# 1. Term frequency (TF) for each doc: tf_counts[doc_id][term] = count
tf_counts = {}
# 2. Document frequency (DF) for each term
df_counts = {}

for doc_id, text in docs.items():
    tokens = preprocess(text)
    tf_counts[doc_id] = {}
    for tok in tokens:
        tf_counts[doc_id][tok] = tf_counts[doc_id].get(tok, 0) + 1

    for tok in set(tokens):
        df_counts[tok] = df_counts.get(tok, 0) + 1

# 3. Compute IDF = log(N / df)
idf_scores = {}
for tok, df in df_counts.items():
    idf_scores[tok] = math.log(N / df)

# 4. Compute TF-IDF = TF * IDF
tfidf_scores = {}
for doc_id in all_doc_ids:
    tfidf_scores[doc_id] = {}
    for tok, tf in tf_counts[doc_id].items():
        tfidf_scores[doc_id][tok] = tf * idf_scores[tok]

# ---------------------------------------------------------------- Part B
def rank_tfidf(query):
    \"\"\"Return sorted list of (doc_id, score) using TF-IDF.\"\"\"
    q_tokens = preprocess(query)
    scores = {}
    for doc_id in all_doc_ids:
        doc_score = 0.0
        for tok in q_tokens:
            doc_score += tfidf_scores[doc_id].get(tok, 0.0)
        if doc_score > 0:
            scores[doc_id] = doc_score

    # Sort by score descending, then by doc_id alphabetically as tie-breaker
    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))

# ---------------------------------------------------------------- Part C
# BM25 Parameters
k1 = 1.2
b = 0.75

# Compute document lengths and average length (avgdl)
doc_lens = {doc_id: len(preprocess(text)) for doc_id, text in docs.items()}
avgdl = sum(doc_lens.values()) / N

def rank_bm25(query):
    \"\"\"Return sorted list of (doc_id, score) using BM25 scoring.\"\"\"
    q_tokens = preprocess(query)
    scores = {}
    for doc_id in all_doc_ids:
        doc_score = 0.0
        doc_len = doc_lens[doc_id]
        for tok in q_tokens:
            if tok in df_counts:
                df = df_counts[tok]
                # BM25 IDF formulation matching Part C
                term_idf = math.log(N / df)
                tf = tf_counts[doc_id].get(tok, 0)

                numerator = tf * (k1 + 1)
                denominator = tf + k1 * (1 - b + b * (doc_len / avgdl))

                doc_score += term_idf * (numerator / denominator)
        if doc_score > 0:
            scores[doc_id] = doc_score

    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))

if __name__ == \"__main__\":
    queries = [\"cat dog\", \"sat mat\", \"played\"]
    for q in queries:
        print(f\"\\nQuery: '{q}'\")
        tfidf_ranks = rank_tfidf(q)[:5]
        bm25_ranks = rank_bm25(q)[:5]

        print(f\"  {'Rank':<4} {'TF-IDF doc':<14} {'score':<7} {'BM25 doc':<14} {'score':<7}\")
        for i in range(5):
            tfidf_doc, tfidf_sc = tfidf_ranks[i] if i < len(tfidf_ranks) else (\"-\", 0.0)
            bm25_doc, bm25_sc = bm25_ranks[i] if i < len(bm25_ranks) else (\"-\", 0.0)
            print(f\"  {i+1:<4} {tfidf_doc:<14} {tfidf_sc:<7.3f} {bm25_doc:<14} {bm25_sc:<7.3f}\")
"""

with open('exercise2/starter.py', 'w', encoding='utf-8') as f:
    f.write(exercise2_code.strip())

print("exercise2/starter.py successfully saved!")

exercise2/starter.py successfully saved!


In [21]:
!{sys.executable} exercise2/starter.py


Query: 'cat dog'
  Rank TF-IDF doc     score   BM25 doc       score  
  1    doc15          3.822   doc15          2.990  
  2    doc09          3.149   doc03          2.385  
  3    doc01          2.773   doc08          1.925  
  4    doc08          2.773   doc01          1.878  
  5    doc19          2.773   doc19          1.878  

Query: 'sat mat'
  Rank TF-IDF doc     score   BM25 doc       score  
  1    doc01          5.404   doc01          4.146  
  2    doc12          3.794   doc12          2.702  
  3    doc20          1.897   doc20          1.794  
  4    doc02          1.609   doc02          1.695  
  5    doc10          1.609   doc17          1.695  

Query: 'played'
  Rank TF-IDF doc     score   BM25 doc       score  
  1    doc14          6.438   doc14          2.740  
  2    doc04          3.219   doc04          2.235  
  3    doc03          1.609   doc13          1.633  
  4    doc13          1.609   doc03          1.576  
  5    -              0.000   -              0

## Exercise 3: Evaluation (P@5, R@5, AP, nDCG@5)

In [28]:
import os

# Writing a complete, self-contained implementation of exercise3/starter.py
exercise3_code = '''import os
import sys
import math

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from exercise1.starter import load_documents
from exercise2.starter import rank_tfidf, rank_bm25

docs = load_documents()
all_doc_ids = sorted(docs.keys())

# Part A
JUDGMENTS = {
    "cat dog": {
        "doc01": 1, "doc03": 2, "doc05": 1, "doc08": 1, "doc09": 1,
        "doc13": 1, "doc15": 2, "doc19": 1, "doc20": 1
    },
    "sat mat": {
        "doc01": 2, "doc12": 2, "doc20": 1, "doc02": 1, "doc10": 1, "doc17": 1
    },
    "played": {
        "doc03": 1, "doc04": 1, "doc13": 1, "doc14": 2
    },
    "warm sleep": {
        "doc01": 2, "doc07": 1, "doc08": 1, "doc11": 1, "doc12": 1, "doc16": 1, "doc19": 1
    },
    "window": {
        "doc01": 2, "doc05": 1, "doc11": 1, "doc18": 1
    }
}

# Part B
def precision_at_k(ranking, judgments, k=5):
    if k <= 0:
        return 0.0
    retrieved_k = ranking[:k]
    relevant_count = sum(1 for d in retrieved_k if judgments.get(d, 0) > 0)
    return relevant_count / k

def recall_at_k(ranking, judgments, k=5):
    retrieved_k = ranking[:k]
    total_relevant = sum(1 for d, rel in judgments.items() if rel > 0)
    if total_relevant == 0:
        return 0.0
    relevant_retrieved = sum(1 for d in retrieved_k if judgments.get(d, 0) > 0)
    return relevant_retrieved / total_relevant

def average_precision(ranking, judgments):
    total_relevant = sum(1 for d, rel in judgments.items() if rel > 0)
    if total_relevant == 0:
        return 0.0

    running_relevant = 0
    ap_sum = 0.0
    for i, doc in enumerate(ranking):
        if judgments.get(doc, 0) > 0:
            running_relevant += 1
            precision_i = running_relevant / (i + 1)
            ap_sum += precision_i

    return ap_sum / total_relevant

def ndcg_at_k(ranking, judgments, k=5):
    if k <= 0:
        return 0.0

    retrieved_k = ranking[:k]
    dcg = 0.0
    for i, doc in enumerate(retrieved_k):
        rel = judgments.get(doc, 0)
        dcg += (2**rel - 1) / math.log2(i + 2)

    all_relevant_gains = sorted([rel for rel in judgments.values() if rel > 0], reverse=True)
    ideal_gains_k = all_relevant_gains[:k]
    idcg = 0.0
    for i, rel in enumerate(ideal_gains_k):
        idcg += (2**rel - 1) / math.log2(i + 2)

    if idcg == 0.0:
        return 0.0
    return dcg / idcg

# Part C
def evaluate():
    systems = {
        "TF-IDF": rank_tfidf,
        "BM25": rank_bm25
    }

    print(f"| {'System':<8} | {'Query':<12} | {'P@5':<5} | {'R@5':<5} | {'AP':<5} | {'nDCG@5':<6} |")
    print("|----------|--------------|-------|-------|-------|--------|")

    for sys_name, rank_func in systems.items():
        p_list, r_list, ap_list, ndcg_list = [], [], [], []
        for q in JUDGMENTS.keys():
            ranking = [doc_id for doc_id, _ in rank_func(q)]
            judg = JUDGMENTS[q]

            p5 = precision_at_k(ranking, judg, 5)
            r5 = recall_at_k(ranking, judg, 5)
            ap = average_precision(ranking, judg)
            ndcg5 = ndcg_at_k(ranking, judg, 5)

            p_list.append(p5)
            r_list.append(r5)
            ap_list.append(ap)
            ndcg_list.append(ndcg5)

            print(f"| {sys_name:<8} | {q:<12} | {p5:<5.3f} | {r5:<5.3f} | {ap:<5.3f} | {ndcg5:<6.3f} |")

        mean_p5 = sum(p_list) / len(p_list)
        mean_r5 = sum(r_list) / len(r_list)
        mean_ap = sum(ap_list) / len(ap_list)
        mean_ndcg5 = sum(ndcg_list) / len(ndcg_list)
        print(f"| {sys_name:<8} | {'MEAN':<12} | {mean_p5:<5.3f} | {mean_r5:<5.3f} | {mean_ap:<5.3f} | {mean_ndcg5:<6.3f} |")
        print("|----------|--------------|-------|-------|-------|--------|")

if __name__ == '__main__':
    evaluate()
'''

with open('exercise3/starter.py', 'w', encoding='utf-8') as f:
    f.write(exercise3_code.strip())

print("exercise3/starter.py successfully saved!")

exercise3/starter.py successfully saved!


In [29]:
!{sys.executable} exercise3/starter.py

| System   | Query        | P@5   | R@5   | AP    | nDCG@5 |
|----------|--------------|-------|-------|-------|--------|
| TF-IDF   | cat dog      | 1.000 | 0.556 | 0.963 | 0.797  |
| TF-IDF   | sat mat      | 1.000 | 0.833 | 1.000 | 1.000  |
| TF-IDF   | played       | 0.800 | 1.000 | 1.000 | 1.000  |
| TF-IDF   | warm sleep   | 0.200 | 0.143 | 0.143 | 0.606  |
| TF-IDF   | window       | 0.200 | 0.250 | 0.250 | 0.658  |
| TF-IDF   | MEAN         | 0.640 | 0.556 | 0.671 | 0.812  |
|----------|--------------|-------|-------|-------|--------|
| BM25     | cat dog      | 1.000 | 0.556 | 0.963 | 1.000  |
| BM25     | sat mat      | 1.000 | 0.833 | 1.000 | 1.000  |
| BM25     | played       | 0.800 | 1.000 | 1.000 | 1.000  |
| BM25     | warm sleep   | 0.200 | 0.143 | 0.143 | 0.606  |
| BM25     | window       | 0.200 | 0.250 | 0.250 | 0.658  |
| BM25     | MEAN         | 0.640 | 0.556 | 0.671 | 0.853  |
|----------|--------------|-------|-------|-------|--------|


## Quick self-tests for Exercise 3 metrics (lecture examples)
Expected: AP = 0.806, nDCG@5 = 0.889 once your functions are implemented.

In [26]:
import importlib.util
spec = importlib.util.spec_from_file_location('ex3', 'exercise3/starter.py')
ex3 = importlib.util.module_from_spec(spec); spec.loader.exec_module(ex3)
ranking = ['D1','D2','D3','D4','D5']
print('AP     =', round(ex3.average_precision(ranking, {'D1':1,'D3':1,'D4':1}), 3), '(expected 0.806)')
print('nDCG@5 =', round(ex3.ndcg_at_k(ranking, {'D1':2,'D3':1,'D4':2}, 5), 3), '(expected 0.889)')

AP     = 0.806 (expected 0.806)
nDCG@5 = 0.889 (expected 0.889)
